# Shamiri survey data QA
Reusable checks for: missing pages, school name mismatches, and admission number mismatches.

### What this notebook does
1. imports `Baseline_Survey.csv`, `Endline_Survey.csv`, `schools.csv`, and `prefilling_data.csv`,.
2. Check for missing pages in each survey.
3. Check for School Name mismatches between surveys and reference`school.csv`
4. Check for Admssion number mismatches  between baseline & endline, baseline & prefilling, prefilling & endline, baseline & endline
5. Generate report for each check in different sheets.
6. Exports the report to an `.xlsx` 

Forms stage 1 of the survey data QA


In [1]:
from pathlib import Path
from datetime import date
import pandas as pd

In [2]:
# --- Config ---
parent_root = Path.cwd()

# data directories
data_directory = Path.cwd().parent/'data'
input_folder = data_directory/'Inputs'

# reports output directory (created if missing)
reports_dir = data_directory / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# expected page counts per survey 
PAGES_THRESHOLD = {
    "baseline": 10,
    "endline": 7,
}

# column names used across checks 
NAME_COL = "School Name"
ADM_COL = "Admission number"

# how many (leftmost) columns to keep when exporting a report to Excel
REPORT_N_COLS = 10

In [3]:
# importing data
schools = pd.read_csv(raw_data/'schools.csv')
baseline = pd.read_csv(raw_data/'Baseline_Survey.csv')
prefill = pd.read_csv(raw_data/'prefilling_data.csv')
endline = pd.read_csv(raw_data/'Endline_Survey.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\imma\\Automation\\Shamiri\\notebooks\\data\\schools.csv'

In [10]:
# check for rows whose page count is below the expected count for each survey
def check_missing_pages(df, min_pages, pages_col="Pages"):
    return df[df[pages_col] < min_pages].copy()

#check forrRows whose school name isn't found in the reference school list
def check_name_mismatch(df, reference_names, name_col=NAME_COL):
    return df[~df[name_col].isin(reference_names)].copy()
    
# remove whitespace and uppercase so minor formatting differences don't cause false mismatches
def _normalize_school(series):
    return series.astype(str).str.strip().str.upper()

# checking for admission number mismatches 
def check_admission_mismatch(df, reference_df, name_col=NAME_COL, adm_col=ADM_COL):
    df_key = _normalize_school(df[name_col])
    ref_key = reference_df[name_col]
    ref_pairs = pd.DataFrame({"_key": ref_key, adm_col: reference_df[adm_col].values})

    merged = df.assign(_key=df_key).merge(
        ref_pairs, on=["_key", adm_col], how="left", indicator=True
    )
    mismatches = merged[merged["_merge"] == "left_only"].drop(columns=["_merge", "_key"])
    return mismatches

## Export function
Generate excel workbook report for the QA isues

In [11]:
def export_reports_to_excel(reports: dict, filename: str, output_dir=reports_dir, n_cols=REPORT_N_COLS, timestamp=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if timestamp:
        filename = f"{date.today().isoformat()}_{filename}"
    output_path = output_dir / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in reports.items():
            trimmed = df.iloc[:, :n_cols]
            safe_name = sheet_name[:31]  # Excel sheet name limit
            trimmed.to_excel(writer, sheet_name=safe_name, index=False)

    print(f"Exported {len(reports)} sheet(s) to {output_path}")
    return output_path

In [15]:
# Reports
#  1. Missing Pages
missing_pages_reports = {
    "baseline_missing_pages": check_missing_pages(baseline, PAGES_THRESHOLD["baseline"]),
    "endline_missing_pages": check_missing_pages(endline, PAGES_THRESHOLD["endline"]),
}

# 2. School Name missmatches
reference_schools = schools["School Name"]

name_mismatch_reports = {
    "baseline_name_mismatch": check_name_mismatch(baseline, reference_schools),
    "endline_name_mismatch": check_name_mismatch(endline, reference_schools),
}

# 3. Admission number miss matches (compared per school)
admission_mismatch_reports = {
    "baseline_vs_prefill": check_admission_mismatch(baseline, prefill),
    "endline_vs_prefill": check_admission_mismatch(endline, prefill),
    "baseline_vs_endline": check_admission_mismatch(baseline, endline),
    "endline_vs_baseline": check_admission_mismatch(endline, baseline),
}

# 4. Combined
all_reports = {**missing_pages_reports, **name_mismatch_reports, **admission_mismatch_reports}
export_reports_to_excel(all_reports, "full_qa_report.xlsx")